In [16]:
# define the Parallel Cognitive Attention Block (PCAB)
class ChannelGate(nn.Module):
    def __init__(self, gate_channels, reduction_ratio=16, num_layers=1):
        super(ChannelGate, self).__init__()
        self.gate_channels = gate_channels
        self.mlp = nn.Sequential(
            Flatten(),
            nn.Linear(gate_channels, gate_channels // reduction_ratio),
            nn.BatchNorm1d(gate_channels // reduction_ratio),
            nn.ReLU(),
            nn.Linear(gate_channels // reduction_ratio, gate_channels)
        )

    def forward(self, x):
        avg_pool = F.avg_pool2d(x, x.size(2), stride=x.size(2))
        max_pool = F.max_pool2d(x, x.size(2), stride=x.size(2))
        out = self.mlp(avg_pool + max_pool)
        out = out.unsqueeze(2).unsqueeze(3).expand_as(x)
        return out
class SpatialGate(nn.Module):
    def __init__(self, gate_channels, number_of_dilation=1, dilation_value=2): # dilation_value=dilation_rate
        super(SpatialGate, self).__init__()
        self.gate_channels = gate_channels
        self.reduced_channel = gate_channels
        # the receptive field of dilated convolution with a kernel size of 3 × 3 and a dilation value of 2 is equal to 5 × 5.
        self.conv = nn.Sequential(
            nn.Conv2d(gate_channels, self.reduced_channel, kernel_size=3, padding=dilation_value, dilation=dilation_value),
            nn.BatchNorm2d(self.reduced_channel),
            nn.ReLU()
            )

    def forward(self, x):
        avg_pool = F.avg_pool2d(x, x.size(2), stride=x.size(2))
        max_pool = F.max_pool2d(x, x.size(2), stride=x.size(2))
        out = self.conv(avg_pool + max_pool)
        out = self.conv(x).expand_as(x)
        return out
class PCAB(nn.Module):
    def __init__(self, gate_channels):
        super(PCAB, self).__init__()
        self.channel_att = ChannelGate(gate_channels)
        self.spatial_att = SpatialGate(gate_channels)

    def forward(self, x):
        att_map = torch.sigmoid(torch.add(self.channel_att(x), self.spatial_att(x)))
        out = x + att_map * x
        return out